# ECHO: Quantum Investigation Notebook
**Participant:** Tabeenah Aslam Hisami  
**Mandatory modules:** 01 (The Gatekeeper), 03 (The Two Observers), 05 (The Damaged Message)  
**Stretch module:** 06 (The Impossible Claim)

This notebook reconstructs the tasks described in the ECHO Participant Manual using Qiskit and the Aer simulator. Each module ends with a one-line statement of what the result proves. Run all cells top to bottom (Runtime → Run all) so that every output, circuit drawing and histogram is saved in the file.

## Setup

In [ ]:
%pip -q install qiskit qiskit-aer matplotlib pylatexenc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.0 MB/s eta 0:00:00


In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from collections import Counter
import math
import matplotlib.pyplot as plt

sim = AerSimulator()
shots = 1000
print("Qiskit simulator ready.")

Qiskit simulator ready.


## Module 01 — The Gatekeeper (mandatory)

### Bronze: always 1, and a fair quantum coin
- `X|0⟩ = |1⟩`, so measuring gives 1 every time.
- `H|0⟩ = (|0⟩ + |1⟩)/√2`, so measurement is 50/50 in the ideal model.

**Prediction:** X → `1` in 100% of shots. H → roughly 500/500 (not exactly, because sampling is random).

In [ ]:
always_one = QuantumCircuit(1, 1)
always_one.x(0)
always_one.measure(0, 0)
print(always_one.draw())
counts_one = sim.run(transpile(always_one, sim), shots=shots).result().get_counts()
print("X on |0>:", counts_one)
assert counts_one.get("1", 0) == shots

fair_coin = QuantumCircuit(1, 1)
fair_coin.h(0)
fair_coin.measure(0, 0)
print(fair_coin.draw())
counts_coin = sim.run(transpile(fair_coin, sim), shots=shots).result().get_counts()
print("H on |0>:", counts_coin)
print("Fraction of 1s:", counts_coin.get("1", 0) / shots)
plot_histogram([counts_one, counts_coin], legend=["X", "H"], title="Module 01 Bronze")
plt.show()

     ┌───┐┌─┐
  q: ┤ X ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0 
X on |0>: {'1': 1000}
     ┌───┐┌─┐
  q: ┤ H ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0 
H on |0>: {'0': 491, '1': 509}
Fraction of 1s: 0.509


**Bronze conclusion:** X creates a certain 1; H creates an ideal fair quantum coin. A finite sample will usually be near, not exactly, 50/50.

### Silver: looks uncertain, but is certain
Apply H twice. Since `H·H = I`, the state returns to `|0⟩`, so the final measurement is always 0.

**Prediction:** `{'0': 1000}`.

In [ ]:
hh = QuantumCircuit(1, 1)
hh.h(0)
hh.h(0)
hh.measure(0, 0)
print(hh.draw())
counts_hh = sim.run(transpile(hh, sim), shots=shots).result().get_counts()
print("H then H:", counts_hh)
assert counts_hh == {"0": shots}
plot_histogram(counts_hh, title="Module 01 Silver: H then H")
plt.show()

     ┌───┐┌───┐┌─┐
  q: ┤ H ├┤ H ├┤M├
     └───┘└───┘└╥┘
c: 1/═══════════╩═
                0 
H then H: {'0': 1000}


**Silver conclusion:** The first H creates a superposition, but the second H reverses it. The final result is certain.

### Gold: make and measure `101` from `000`
Qiskit prints the highest-index classical bit on the left. Applying X to qubits 0 and 2 produces `101`.

**Prediction:** `{'101': 1000}`.

In [ ]:
vault = QuantumCircuit(3, 3)
vault.x(0)
vault.x(2)
vault.measure([0, 1, 2], [0, 1, 2])
print(vault.draw())
vault_counts = sim.run(transpile(vault, sim), shots=shots).result().get_counts()
print("Vault code:", vault_counts)
assert vault_counts == {"101": shots}
plot_histogram(vault_counts, title="Module 01 Gold: vault code")
plt.show()

     ┌───┐   ┌─┐   
q_0: ┤ X ├───┤M├───
     └───┘┌─┐└╥┘   
q_1: ─────┤M├─╫────
     ┌───┐└╥┘ ║ ┌─┐
q_2: ┤ X ├─╫──╫─┤M├
     └───┘ ║  ║ └╥┘
c: 3/══════╩══╩══╩═
           1  0  2 
Vault code: {'101': 1000}


**Module 01 — one-line proof:** Basic gates can create deterministic bits, a balanced superposition, and the exact pattern `101`; the two-H sequence shows a superposition can be reversed.

---

## Module 03 — The Two Observers (mandatory)

### Bronze: a pair that always agrees
Create a Bell pair: H on the first qubit, then CNOT. In the Z basis the outcomes are `00` or `11`, never `01` or `10`.

**Prediction:** only `00` and `11`, each about 50%.

In [ ]:
bell = QuantumCircuit(2, 2)
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])
print(bell.draw())
bell_counts = sim.run(transpile(bell, sim), shots=shots).result().get_counts()
print("Bell pair, Z basis:", bell_counts)
assert set(bell_counts).issubset({"00", "11"})
plot_histogram(bell_counts, title="Module 03 Bronze: Bell pair, Z basis")
plt.show()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 
Bell pair, Z basis: {'11': 510, '00': 490}


### Silver: compare with a pre-agreed secret coin
A classical model also produces perfect Z-basis agreement: pick one shared random bit and hand the same bit to both observers. So agreement alone does not prove entanglement.

### Gold: change the measurement basis
Applying H to both qubits before measurement switches both observers to the X basis. The ideal Bell pair still agrees. The simple classical shared-bit model (an equal mixture of `00` and `11`) becomes uncorrelated.

**Prediction:** Bell pair in the X basis → only `00`/`11`. Classical mixture in the X basis → all four outcomes, roughly 25% each.

*Limitation:* this separates the Bell state from this particular classical model only. It is not a full Bell-inequality test against all local hidden-variable models.

In [ ]:
# Quantum Bell pair measured in the X basis (H on both qubits before measuring)
bell_x = QuantumCircuit(2, 2)
bell_x.h(0)
bell_x.cx(0, 1)
bell_x.h(0)
bell_x.h(1)
bell_x.measure([0, 1], [0, 1])
print(bell_x.draw())
bell_x_counts = sim.run(transpile(bell_x, sim), shots=shots).result().get_counts()
print("Bell pair, X basis:", bell_x_counts)
assert set(bell_x_counts).issubset({"00", "11"})

# Classical shared secret bit: prepare |00> or |11> with equal probability,
# then measure both in the X basis (H before measuring).
mix_counts = Counter()
for prep in (0, 1):
    qc = QuantumCircuit(2, 2)
    if prep == 1:
        qc.x(0)
        qc.x(1)
    qc.h(0)
    qc.h(1)
    qc.measure([0, 1], [0, 1])
    mix_counts.update(sim.run(transpile(qc, sim), shots=shots // 2).result().get_counts())
mix_counts = dict(mix_counts)
print("Classical shared-bit mixture, X basis:", mix_counts)

plot_histogram([bell_x_counts, mix_counts],
               legend=["Bell pair", "Classical shared bit"],
               title="Module 03 Gold: X-basis measurement")
plt.show()

     ┌───┐     ┌───┐┌─┐   
q_0: ┤ H ├──■──┤ H ├┤M├───
     └───┘┌─┴─┐├───┤└╥┘┌─┐
q_1: ─────┤ X ├┤ H ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 
Bell pair, X basis: {'00': 481, '11': 519}
Classical shared-bit mixture, X basis: {'01': 225, '10': 249, '00': 295, '11': 231}


**Module 03 — one-line proof:** Perfect agreement in one basis is not enough to prove a quantum link; the Bell pair keeps its agreement when the basis changes, while a simple shared-bit model does not.

---

## Module 05 — The Damaged Message (mandatory)

The manual provides 32 triples. Decode each triple by majority vote: the bit appearing at least twice is the recovered bit.

In [ ]:
corrupted = """
100 011 001 001 001 011 100 110 001 011 000 000 001 100 011 011
000 111 000 000 110 000 001 000 000 111 001 010 101 011 111 111
""".split()

assert len(corrupted) == 32
decoded_bits = ''.join('1' if t.count('1') >= 2 else '0' for t in corrupted)
bytes8 = [decoded_bits[i:i+8] for i in range(0, len(decoded_bits), 8)]
message = ''.join(chr(int(b, 2)) for b in bytes8)

print("Triples:", len(corrupted))
print("Recovered bits:", decoded_bits)
print("Bytes:", bytes8)
print("Recovered message:", message)
assert message == "ECHO"

Triples: 32
Recovered bits: 01000101010000110100100001001111
Bytes: ['01000101', '01000011', '01001000', '01001111']
Recovered message: ECHO


### Silver: show when majority vote fails
The scheme assumes at most one flip in each triple. If the original bit is `0` but two of its copies flip, `000 → 011`; majority vote returns `1`, which is wrong.

In [ ]:
original = "000"
two_flips = "011"
majority_decodes_to = '1' if two_flips.count('1') >= 2 else '0'
print(f"Original {original}; received {two_flips}; decoded {majority_decodes_to}")
assert majority_decodes_to == "1" and majority_decodes_to != original[0]

Original 000; received 011; decoded 1


### Gold: correction versus detection, and the qubit limitation
- A three-bit repetition code can **correct one bit flip** per triple.
- It can **detect up to two bit flips** by noticing that the triple is not all equal, but it cannot always identify the original bit when two flips occur.
- An unknown quantum state cannot simply be copied three times because of the **no-cloning theorem**. Quantum error correction uses carefully encoded multi-qubit states and measurements of error syndromes instead of making identical copies of an arbitrary unknown qubit.

**Module 05 — one-line proof:** Majority vote recovers `ECHO` from the provided stream under the stated one-flip-per-triple assumption, but two flips in a triple can defeat it.

---

## Module 06 — The Impossible Claim (stretch)

Quantum teleportation transfers an unknown qubit state using a shared entangled pair plus **two ordinary classical bits**. The receiver (Bob) cannot recover the state without those bits.

We teleport |ψ⟩ = cos(θ/2)|0⟩ + sin(θ/2)|1⟩ with θ = π/3. Alice's input is on q0, the shared Bell pair is q1–q2, and Bob holds q2. The classical bits are kept separate: c0 and c1 are Alice's results, c2 is Bob's final measurement.

**Predictions**
1. With the corrections applied, Bob's P(1) = sin²(θ/2) = 0.25.
2. With the corrections removed (the classical bits never arrive), Bob's qubit is maximally mixed, so P(1) = 0.50. That is the same as a coin flip and carries no message.

In [ ]:
theta = math.pi / 3

def teleport_circuit(apply_corrections=True):
    qc = QuantumCircuit(3, 3)
    qc.ry(theta, 0)          # unknown input state on q0
    qc.h(1)
    qc.cx(1, 2)              # shared Bell pair q1-q2
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)         # Alice's classical bit c0
    qc.measure(1, 1)         # Alice's classical bit c1
    if apply_corrections:
        with qc.if_test((qc.clbits[1], 1)):
            qc.x(2)
        with qc.if_test((qc.clbits[0], 1)):
            qc.z(2)
    qc.measure(2, 2)         # Bob's result in c2
    return qc

def bob_p1(counts):
    total = sum(counts.values())
    return sum(v for k, v in counts.items() if k[0] == "1") / total  # leftmost bit = c2

tele = teleport_circuit(True)
print(tele.draw())
tele_counts = sim.run(transpile(tele, sim), shots=4000).result().get_counts()
print("With corrections (bits shown as c2 c1 c0):", tele_counts)
print("Bob P(1) with corrections:", round(bob_p1(tele_counts), 3), "| expected", round(math.sin(theta/2)**2, 3))

no_corr = teleport_circuit(False)
nc_counts = sim.run(transpile(no_corr, sim), shots=4000).result().get_counts()
print("Without corrections:", nc_counts)
print("Bob P(1) without corrections:", round(bob_p1(nc_counts), 3), "| expected 0.5")

plot_histogram([tele_counts, nc_counts], legend=["With corrections", "No classical bits"],
               title="Module 06: teleportation", figsize=(9, 4))
plt.show()

     ┌─────────┐          ┌───┐┌─┐                                          »
q_0: ┤ Ry(π/3) ├───────■──┤ H ├┤M├──────────────────────────────────────────»
     └──┬───┬──┘     ┌─┴─┐└┬─┬┘└╥┘                                          »
q_1: ───┤ H ├─────■──┤ X ├─┤M├──╫───────────────────────────────────────────»
        └───┘   ┌─┴─┐└───┘ └╥┘  ║   ┌──────  ┌───┐ ───────┐   ┌──────  ┌───┐»
q_2: ───────────┤ X ├───────╫───╫───┤ If-0  ─┤ X ├  End-0 ├───┤ If-0  ─┤ Z ├»
                └───┘       ║   ║   └──╥───  └───┘ ───────┘   └──╥───  └───┘»
                            ║   ║ ┌────╨────┐               ┌────╨────┐     »
c: 3/═══════════════════════╩═══╩═╡ c_1=0x1 ╞═══════════════╡ c_0=0x1 ╞═════»
                            1   0 └─────────┘               └─────────┘     »
«                  
«q_0: ─────────────
«                  
«q_1: ─────────────
«      ───────┐ ┌─┐
«q_2:   End-0 ├─┤M├
«      ───────┘ └╥┘
«c: 3/═══════════╩═
«                2 
With corrections (bits shown as c2 c1 c0

### What if the classical bits never arrive?
Without Alice's two bits, Bob does not know which correction to apply. Averaged over Alice's four possible outcomes, Bob's state is maximally mixed (P(1) = 0.5 above), no matter what Alice prepared. The shared entanglement alone transmits neither the chosen state nor a controllable signal.

**Module 06 — one-line proof:** Teleportation reproduces the input state only when Alice's two classical bits are sent and Bob applies the matching corrections, so it is not faster-than-light messaging.

---

---

# Evidence Board

| FACT — directly observed or computed | CLAIM — stated by a person or log | INFERENCE — interpretation | OPEN QUESTION — unresolved |
|---|---|---|---|
| Module 01 produces `1`, about 50/50, `0` after H-H, and `101` for the vault circuit. | Zara claimed ECHO sent a message faster than light. | The elementary gates behave as quantum theory predicts; this alone does not support faster-than-light signalling. | What exactly did Zara's original hardware display, and were its logs intact? |
| The Bell circuit produces only `00`/`11` in the Z basis and agrees in the X basis. | Vance said ECHO was never real; Sana said she deleted a corrupted log; Bilal said he warned Zara; ARIA said no unauthorised command was issued. | Agreement alone can be explained by a shared classical bit; changing measurement settings reveals stronger quantum correlations. | Which person, if any, altered the experiment or its records? |
| Majority vote decodes the provided triples as `ECHO`. | The manual says at most one bit in each triple was flipped in transit. | Under that assumption, the repetition code recovers the message. | Was the one-flip assumption true for the real machine? |
| Teleportation requires classical measurement results for the receiver's corrections. | Zara claimed a message was teleported with no qubit crossing the gap. | Standard teleportation does not enable faster-than-light communication. | What did Zara mean by “message,” and what physical measurements were actually performed? |

## Verdict (evidence-based, provisional)

**Verdict:** ECHO may have demonstrated quantum correlations or a teleportation-like protocol, but the claim that it sent a controllable message faster than light is not supported by the evidence in this manual. Entanglement produces correlations; it does not let one observer choose the other observer's result or communicate without a classical channel.

**Confidence:** High in the physics conclusion that entanglement alone cannot send a faster-than-light message; low in any conclusion about who caused Zara's disappearance, because the manual gives suspect statements but no verified forensic logs identifying a culprit.

**Strongest competing theory:** Zara honestly misinterpreted correlations as communication and convinced herself the experiment was a breakthrough. This fits the physics evidence without requiring sabotage, but it does not explain her disappearance.

**One limitation:** This notebook recreates the manual's stated tasks; it does not inspect the original Colab's hidden seed circuit, machine logs, or physical apparatus.

**One next experiment:** Run a controlled Bell-test experiment with independently chosen measurement settings, sufficient repeated trials, and a documented chain of custody for the data. This would test the correlation claim more rigorously, although it would still not establish faster-than-light signalling.

---

# Five-minute case presentation

**Opening (30 seconds):** “I reconstructed the key parts of ECHO, and tested deterministic and probabilistic gates, compared quantum correlations with a classical shared-bit model, decoded the damaged message, and examined teleportation.”

**Evidence (90 seconds):** “Module 01 produced a certain 1 with X, a near 50/50 result with H, a certain 0 with H-H, and the target pattern 101. Module 03 produced agreement in the Z basis, but agreement alone is not proof of entanglement. Changing to the X basis distinguishes the Bell pair from the simple classical shared-bit model. Module 05 decodes the 32 triples to ECHO by majority vote, under the one-flip-per-triple assumption. Module 06 shows that teleportation needs two classical bits for the receiver's corrections.”

**Crux (60 seconds):** “Entanglement is not a communication channel. Each local observer's result is not controllable in a way that sends a chosen message. The correlations become meaningful when results are compared, and comparison requires ordinary communication.”

**Verdict and competing theory (60 seconds):** “We reject the faster-than-light messaging claim based on the reconstructed protocol. Our strongest alternative is that Zara mistook correlation for communication. We cannot conclude who caused her disappearance because the manual contains no decisive forensic evidence.”

**Limitation and next test (30 seconds):** “We did not access the original machine logs or hidden notebook cells. Our next step would be a carefully controlled Bell test with independently selected measurement settings and documented data integrity.”

## Submission checklist
- [ ] Run every cell top to bottom (Runtime → Run all) and confirm there are no errors.
- [ ] Save the notebook **with outputs** (circuit drawings, counts and histograms visible).
- [ ] Push the notebook and the case report to the GitHub repository.
- [ ] Open the repository in a private/incognito window to confirm both files are viewable.
- [ ] Be ready to explain why H·H returns |0⟩, why agreement alone is insufficient, why majority vote can fail, and why teleportation needs classical bits.